In [1]:
# Opening-Day null details are checked after `lineup` is built below.

In [2]:
import polars as pl
from Python import config
from Python.pipeline import training

pitcher_rolling = pl.read_parquet(config.PITCHER_ROLLING_PATH)
batter_rolling = pl.read_parquet(config.BATTER_ROLLING_PATH)
park_factors = (
    pl.read_parquet(config.PARK_FACTORS_PATH)
    if config.PARK_FACTORS_PATH.exists()
    else None
)
pitcher_rolling.shape, batter_rolling.shape, (park_factors.shape if park_factors is not None else None)

((14124, 236), (146385, 21), (120, 5))

In [3]:
print(training._LINEUP_RATE_COLUMNS)

{'opp_lineup_whiff': 'whiff_rate_std', 'opp_lineup_chase': 'chase_rate_std'}


In [4]:
lineup = training.opposing_lineup_features(pitcher_rolling, batter_rolling)
lineup.shape
lineup.head()

game_pk,pitcher,opp_lineup_size,opp_lineup_k,opp_lineup_k_vs_hand,opp_lineup_whiff,opp_lineup_chase
i64,i64,u32,f64,f64,f64,f64
778409,680736,9,0.23821,0.379118,0.109125,0.296875
747070,519242,9,0.194205,0.177325,0.106995,0.257368
745429,543294,9,0.181009,0.184165,0.084799,0.273418
746038,607259,9,0.234825,0.231724,0.128462,0.316098
746330,676979,9,0.275478,0.211111,0.13884,0.313261


In [5]:
check = pl.DataFrame({
    "pitcher_rolling_rows": [pitcher_rolling.height],
    "lineup_rows": [lineup.height],
    "unique_game_pitcher_pairs": [pitcher_rolling.select("game_pk", "pitcher").unique().height],
})
check

pitcher_rolling_rows,lineup_rows,unique_game_pitcher_pairs
i64,i64,i64
14124,14124,14124


In [6]:
print(lineup.select(pl.all().exclude("game_pk", "pitcher").null_count()))

# These rows are expected season openers: no batter has prior season-to-date PA.
opening_nulls = lineup.filter(pl.col("opp_lineup_k").is_null()).join(
    pitcher_rolling.select("game_pk", "pitcher", "opp_team", "game_date"),
    on=["game_pk", "pitcher"],
)
opening_nulls.select("game_date").unique().sort("game_date"), opening_nulls.height

shape: (1, 5)
┌─────────────────┬──────────────┬──────────────────────┬──────────────────┬──────────────────┐
│ opp_lineup_size ┆ opp_lineup_k ┆ opp_lineup_k_vs_hand ┆ opp_lineup_whiff ┆ opp_lineup_chase │
│ ---             ┆ ---          ┆ ---                  ┆ ---              ┆ ---              │
│ u32             ┆ u32          ┆ u32                  ┆ u32              ┆ u32              │
╞═════════════════╪══════════════╪══════════════════════╪══════════════════╪══════════════════╡
│ 0               ┆ 90           ┆ 99                   ┆ 90               ┆ 90               │
└─────────────────┴──────────────┴──────────────────────┴──────────────────┴──────────────────┘


(shape: (6, 1)
 ┌────────────┐
 │ game_date  │
 │ ---        │
 │ date       │
 ╞════════════╡
 │ 2023-03-30 │
 │ 2024-03-28 │
 │ 2024-03-29 │
 │ 2025-03-18 │
 │ 2025-03-27 │
 │ 2025-03-28 │
 └────────────┘,
 90)

In [7]:
pitcher_training = training.build_pitcher_training(pitcher_rolling, batter_rolling, park_factors)
pitcher_training.shape
pitcher_training.columns

['game_pk',
 'pitcher',
 'game_date',
 'season',
 'player_name',
 'home_team',
 'away_team',
 'p_throws',
 'PA',
 'K',
 'Outs',
 'is_home',
 'opp_team',
 'k_rate',
 'pitcher_name',
 'k_rate_P5',
 'k_rate_P10',
 'k_rate_P20',
 'k_rate_std',
 'bb_rate_P5',
 'bb_rate_P10',
 'bb_rate_P20',
 'bb_rate_std',
 'csw_rate_P5',
 'csw_rate_P10',
 'csw_rate_P20',
 'csw_rate_std',
 'swstr_rate_P5',
 'swstr_rate_P10',
 'swstr_rate_P20',
 'swstr_rate_std',
 'whiff_rate_P5',
 'whiff_rate_P10',
 'whiff_rate_P20',
 'whiff_rate_std',
 'cs_rate_P5',
 'cs_rate_P10',
 'cs_rate_P20',
 'cs_rate_std',
 'chase_rate_P5',
 'chase_rate_P10',
 'chase_rate_P20',
 'chase_rate_std',
 'zone_rate_P5',
 'zone_rate_P10',
 'zone_rate_P20',
 'zone_rate_std',
 'contact_rate_P5',
 'contact_rate_P10',
 'contact_rate_P20',
 'contact_rate_std',
 'gb_rate_P5',
 'gb_rate_P10',
 'gb_rate_P20',
 'gb_rate_std',
 'hr_rate_P5',
 'hr_rate_P10',
 'hr_rate_P20',
 'hr_rate_std',
 'ff_velo_P3',
 'ff_velo_P5',
 'ff_velo_P10',
 'ff_spinrate_P3

In [8]:
pitcher_training_no_park = training.build_pitcher_training(pitcher_rolling, batter_rolling, park_factors=None)
"park_k_factor" in pitcher_training_no_park.columns

False

In [9]:
lineup_source_cols = {"k_rate_std", "k_rate_std_vL", "k_rate_std_vR", *training._LINEUP_RATE_COLUMNS.values()}
non_rolling_leak_risk = [c for c in lineup_source_cols if not c.endswith(("_std", "_std_vL", "_std_vR"))]
non_rolling_leak_risk

[]

In [10]:
if park_factors is not None:
    spot = (
        pitcher_training
        .filter(pl.col("season") == 2025)
        .select("game_pk", "pitcher", "home_team", "season", "park_k_factor")
        .head(5)
    )
    park_factors.filter(pl.col("season") == 2025).select("home_team", "park_k_factor").head(5)

In [11]:
paths = training.run()
paths

[level 3] wrote pitcher_training: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\Data\processed\pitcher_training.parquet
[level 3] wrote batter_training: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\Data\processed\batter_training.parquet


{'pitcher_training': WindowsPath('C:/Users/ckaplinger/Downloads/Personal-Projects/MLB-Props/Data/processed/pitcher_training.parquet'),
 'batter_training': WindowsPath('C:/Users/ckaplinger/Downloads/Personal-Projects/MLB-Props/Data/processed/batter_training.parquet')}

In [12]:
pitcher_training_final = pl.read_parquet(config.PITCHER_TRAINING_PATH)
batter_training_final = pl.read_parquet(config.BATTER_TRAINING_PATH)

label_cols = ["K", "PA", "Outs", "k_rate"]
[c for c in label_cols if c in pitcher_training_final.columns]

['K', 'PA', 'Outs', 'k_rate']